In [6]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
from pathlib import Path

class MILBagDataset(Dataset):
    def __init__(self, mfcc_root, prior_csv, bag_csv):
        self.mfcc_root = Path(mfcc_root)

        self.prior_df = pd.read_csv(prior_csv)
        self.bag_df = pd.read_csv(bag_csv)

        # 构建索引（每个bag一条）
        self.index = self.bag_df[["file_num", "bag_idx", "label"]].drop_duplicates()

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        row = self.index.iloc[idx]

        file_num = int(row["file_num"])
        bag_idx = int(row["bag_idx"])
        label = float(row["label"])

        # -----------------------
        # 1. 读取 MFCC
        # -----------------------
        mfcc_path = self.mfcc_root / f"file_{file_num:02d}" / f"bag_{bag_idx}.npy"
        x = np.load(mfcc_path)  # (60, D)

        # -----------------------
        # 2. 读取 prior
        # -----------------------
        prior = self.prior_df[
            (self.prior_df.file_num == file_num) &
            (self.prior_df.bag_idx == bag_idx)
        ].sort_values("instance_idx")["prior_score"].values

        # 安全检查（关键）
        assert len(prior) == 60, f"prior length != 60: file {file_num} bag {bag_idx}"

        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(prior, dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32)
        )

In [7]:
# TCN
import torch.nn as nn

class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation):
        super().__init__()

        padding = (kernel_size - 1) * dilation

        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size,
                              padding=padding,
                              dilation=dilation)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)

        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None

    def forward(self, x):
        out = self.conv(x)

        # 裁剪 padding（保证因果结构）
        out = out[:, :, :-self.conv.padding[0]]

        out = self.relu(out)
        out = self.dropout(out)

        res = x if self.downsample is None else self.downsample(x)

        return out + res

In [8]:
# TCN Encoder
class TCNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()

        self.network = nn.Sequential(
            TemporalBlock(input_dim, hidden_dim, kernel_size=3, dilation=1),
            TemporalBlock(hidden_dim, hidden_dim, kernel_size=3, dilation=2),
            TemporalBlock(hidden_dim, hidden_dim, kernel_size=3, dilation=4),
        )

    def forward(self, x):
        """
        x: [B, 60, D]
        """
        x = x.transpose(1, 2)   # → [B, D, 60]
        out = self.network(x)
        out = out.transpose(1, 2)  # → [B, 60, H]
        return out

In [9]:
#Prioe-Guided MIL
import torch.nn.functional as F

class PriorTCNMIL(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, alpha=1.0):
        super().__init__()

        self.alpha = alpha

        self.encoder = TCNEncoder(input_dim, hidden_dim)

        self.attn_fc = nn.Linear(hidden_dim, 1)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x, prior):
        """
        x: [B, 60, D]
        prior: [B, 60]
        """

        # -----------------------
        # TCN feature
        # -----------------------
        h = self.encoder(x)   # [B, 60, H]

        # -----------------------
        # Attention
        # -----------------------
        attn_logits = self.attn_fc(h).squeeze(-1)  # [B, 60]

        # 🔥 融合 prior（你的核心创新）
        attn_logits = attn_logits + self.alpha * prior

        attn = F.softmax(attn_logits, dim=1)

        # -----------------------
        # Bag representation
        # -----------------------
        z = torch.sum(h * attn.unsqueeze(-1), dim=1)

        # -----------------------
        # Classification
        # -----------------------
        y = torch.sigmoid(self.classifier(z)).squeeze(-1)

        return y, attn

In [10]:
# 训练骨架
from torch.utils.data import DataLoader

dataset = MILBagDataset(
    mfcc_root=r"D:\Project_Github\audio_click_mil\processed_data\mfcc",
    prior_csv=r"D:\Project_Github\audio_click_mil\processed_data\instance_prior.csv",
    bag_csv=r"D:\Project_Github\audio_click_mil\processed_data\all_bags.csv"
)

loader = DataLoader(dataset, batch_size=8, shuffle=True)

model = PriorTCNMIL(input_dim=40, hidden_dim=128, alpha=1.0)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

for epoch in range(20):
    for x, prior, y in loader:
        pred, attn = model(x, prior)

        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} Loss: {loss.item():.4f}")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Project_Github\\audio_click_mil\\processed_data\\mfcc\\file_24\\bag_9.npy'